In [5]:
import os
import sys
from ctypes import (
    c_int,
    c_uint,
    c_byte,
    c_ubyte,
    c_short,
    c_double,
    c_float,
    c_long,
    POINTER,
    c_char_p,
)
from ctypes import pointer, byref, windll, cdll, create_string_buffer

# Win + Python 3.8+: 동일 폴더 의존 DLL 검색
_shamrock_dir = r"C:\Program Files\Andor SDK\Shamrock64"
if sys.platform == "win32" and os.path.isdir(_shamrock_dir):
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(_shamrock_dir)
    os.environ["PATH"] = _shamrock_dir + os.pathsep + os.environ.get("PATH", "")

lib0 = windll.LoadLibrary(r"C:\Program Files\Andor SDK\Shamrock64\atshamrock.dll")
andorlibpath = r"C:\Program Files\Andor SDK\Shamrock64\ShamrockCIF.dll"
lib = windll.LoadLibrary(andorlibpath)


def _bind_shamrock_cif_prototypes(lib):
    """argtypes 없이 byref(포인터)를 넘기면 호출 규약이 깨져 P1INVALID 등이 날 수 있음 (ShamrockCIF.h 기준)."""
    lib.ShamrockInitialize.argtypes = [c_char_p]
    lib.ShamrockInitialize.restype = c_uint
    lib.ShamrockClose.argtypes = []
    lib.ShamrockClose.restype = c_uint
    lib.ShamrockGetNumberDevices.argtypes = [POINTER(c_int)]
    lib.ShamrockGetNumberDevices.restype = c_uint
    lib.ShamrockGetSerialNumber.argtypes = [c_int, c_char_p]
    lib.ShamrockGetSerialNumber.restype = c_uint
    lib.ShamrockEepromGetOpticalParams.argtypes = [
        c_int,
        POINTER(c_float),
        POINTER(c_float),
        POINTER(c_float),
    ]
    lib.ShamrockEepromGetOpticalParams.restype = c_uint
    lib.ShamrockGetGrating.argtypes = [c_int, POINTER(c_int)]
    lib.ShamrockGetGrating.restype = c_uint
    lib.ShamrockSetGrating.argtypes = [c_int, c_int]
    lib.ShamrockSetGrating.restype = c_uint
    lib.ShamrockSetWavelength.argtypes = [c_int, c_float]
    lib.ShamrockSetWavelength.restype = c_uint
    lib.ShamrockGetWavelength.argtypes = [c_int, POINTER(c_float)]
    lib.ShamrockGetWavelength.restype = c_uint


_bind_shamrock_cif_prototypes(lib)


In [6]:
SHAMROCK_COMMUNICATION_ERROR = 20201
SHAMROCK_SUCCESS = 20202
SHAMROCK_P1INVALID = 20266
SHAMROCK_P2INVALID = 20267
SHAMROCK_P3INVALID = 20268
SHAMROCK_P4INVALID = 20269
SHAMROCK_P5INVALID = 20270
SHAMROCK_NOT_INITIALIZED = 20275
SHAMROCK_NOT_AVAILABLE = 20292

SHAMROCK_ACCESSORYMIN = 0
SHAMROCK_ACCESSORYMAX = 1
SHAMROCK_FILTERMIN = 1
SHAMROCK_FILTERMAX = 6
SHAMROCK_TURRETMIN = 1
SHAMROCK_TURRETMAX = 3
SHAMROCK_GRATINGMIN = 1
SHAMROCK_SLITWIDTHMIN = 10
SHAMROCK_SLITWIDTHMAX = 2500
SHAMROCK_I24SLITWIDTHMAX = 24000
SHAMROCK_SHUTTERMODEMIN = 0
SHAMROCK_SHUTTERMODEMAX = 1
SHAMROCK_DET_OFFSET_MIN = -240000
SHAMROCK_DET_OFFSET_MAX = 240000
SHAMROCK_GRAT_OFFSET_MIN = -20000
SHAMROCK_GRAT_OFFSET_MAX = 20000

SHAMROCK_SLIT_INDEX_MIN  = 1
SHAMROCK_SLIT_INDEX_MAX  = 4

SHAMROCK_INPUT_SLIT_SIDE  = 1
SHAMROCK_INPUT_SLIT_DIRECT  = 2
SHAMROCK_OUTPUT_SLIT_SIDE  = 3
SHAMROCK_OUTPUT_SLIT_DIRECT = 4

SHAMROCK_FLIPPER_INDEX_MIN  = 1
SHAMROCK_FLIPPER_INDEX_MAX  = 2
SHAMROCK_PORTMIN = 0
SHAMROCK_PORTMAX = 1

SHAMROCK_INPUT_FLIPPER   = 1
SHAMROCK_OUTPUT_FLIPPER = 2
SHAMROCK_DIRECT_PORT = 0
SHAMROCK_SIDE_PORT = 1

SHAMROCK_ERRORLENGTH = 64

consts_by_num = dict()
for name,num in list(locals().items()):
    if name.startswith("SHAMROCK_"):
        consts_by_num[num] = name

In [7]:
def _err(retval):
    if retval == SHAMROCK_SUCCESS:
        return retval
    else:
        err_name = consts_by_num.get(retval)
        raise IOError( "Andor SHAMROCK Failure {}: {}".format(retval, err_name))


In [8]:
_err(lib.ShamrockInitialize(b""))  # Python 3: c_char_p 인자는 bytes

20202

In [9]:
n_dev = c_int()
_err(lib.ShamrockGetNumberDevices(byref(n_dev)))
n = n_dev.value

x = create_string_buffer(64)

for i in range(n):
    _err(lib.ShamrockGetSerialNumber(i,x))
    print(i,x.value)
    

In [10]:
dev_id = 0

f = c_float()
ang = c_float()
tilt = c_float()

_err(lib.ShamrockEepromGetOpticalParams(dev_id, byref(f), byref(ang), byref(tilt)))

print("ShamrockEepromGetOpticalParams", f.value, ang.value, tilt.value)

OSError: Andor SHAMROCK Failure 20266: SHAMROCK_P1INVALID

In [ ]:
x = c_int()
lib.ShamrockGetGrating(0, byref(x))
x.value

1

In [ ]:
_err(lib.ShamrockSetWavelength(dev_id, c_float(500.)))

20202

In [ ]:
lib.ShamrockClose()

20202

In [ ]:
x = c_float()
lib.ShamrockGetWavelength(0, byref(x))
x.value

500.0

In [ ]:
_err(lib.ShamrockSetGrating(dev_id, 2))

20202